# Notebook 63 — Paper Synthesis: Governing Field, Symbolic Chart, and Universality Boundary

Notebook 63 consolidates the zeta-constraint-lab result into a paper-ready synthesis.

The modeling arc from Notebooks 41–62 supports a clear conclusion:

> The residual dynamics are governed by a shared low-order field in `(r,c)`, while regime variation is captured by a sparse symbolic coordinate chart for the coefficient vector. Smooth latent ODE transport is not the right final abstraction.

## Final model structure

### Layer 1 — shared residual-flow field

\[
g(r,c;\beta)
=
\beta_0
+\beta_c c
+\beta_r r
+\beta_{c^3}c^3
+\beta_{r^2}r^2
+\beta_{rc^2}rc^2
\]

### Layer 2 — symbolic coefficient chart

\[
\beta
=
f(\text{forcing mode},\text{flow mode},\text{system},\text{task},k)
\]

## Notebook goals

1. Rebuild the coefficient table and symbolic chart.
2. Produce final paper-level figures:
   - governing template
   - coefficient manifold
   - term stability
   - universality boundary
   - minimal chart tradeoff
3. Compare full symbolic, pruned symbolic, ridge, and latent transport baselines at summary level.
4. Generate paper-ready tables:
   - final equations
   - stable terms
   - block verdicts
5. Export summary CSV/Markdown artifacts.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut
from sklearn.decomposition import PCA

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

np.random.seed(42)

OUTPUT_DIR = "paper_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

## Rebuild shared governing template and coefficient table

In [ ]:
TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "template_rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        errs.append(math.sqrt(mean_squared_error(integrate(beta_ref, r0), integrate(beta_cmp, r0))))
    return float(np.mean(errs))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())
coef_df.to_csv(os.path.join(OUTPUT_DIR, "coefficient_table.csv"), index=False)

## Figure 1 — Coefficient manifold and regime organization

In [ ]:
coef_scaler = StandardScaler()
Y = coef_df[COEF_COLS].to_numpy(dtype=float)
Ystd = coef_scaler.fit_transform(Y)

pca = PCA(n_components=min(6, len(COEF_COLS)), random_state=42)
Z = pca.fit_transform(Ystd)

coef_df["pc1"] = Z[:, 0]
coef_df["pc2"] = Z[:, 1]
coef_df["pc3"] = Z[:, 2] if Z.shape[1] > 2 else 0.0

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Cumulative explained variance:", np.cumsum(pca.explained_variance_ratio_))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["forcing_mode", "flow_mode", "system"]):
    for val in sorted(coef_df[col].astype(str).unique()):
        sub = coef_df[coef_df[col].astype(str) == val]
        ax.scatter(sub["pc1"], sub["pc2"], label=val, s=45)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"Coefficient manifold by {col}")
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_coefficient_manifold.png"), dpi=180, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, len(pca.explained_variance_ratio_) + 1), np.cumsum(pca.explained_variance_ratio_), marker="o")
ax.set_xlabel("component")
ax.set_ylabel("cumulative explained variance")
ax.set_title("Coefficient manifold cumulative variance")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1b_pca_variance.png"), dpi=180, bbox_inches="tight")
plt.show()

## Symbolic chart construction

In [ ]:
def build_symbolic_features(df_in, columns=None, reduced_terms=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)

    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")

    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")

    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if reduced_terms is not None:
        X = X.reindex(columns=reduced_terms, fill_value=0.0)

    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)

    return X.astype(float)

def format_equation(name, term_list, intercept):
    parts = [f"{name} = {intercept:.5f}"]
    for feat, val in term_list:
        sign = "+" if val >= 0 else "-"
        parts.append(f"{sign} {abs(val):.5f}·{feat}")
    return " ".join(parts)

## Figure 2 — Term stability and minimal symbolic chart

In [ ]:
def term_stability_table(df_in, n_splits=8, threshold=1e-4):
    n = len(df_in)
    if n <= 12:
        splitter = LeaveOneOut()
    else:
        splitter = KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)

    all_features = build_symbolic_features(df_in).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in COEF_COLS}
    fold_count = 0

    for train_idx, test_idx in splitter.split(df_in):
        train_df = df_in.iloc[train_idx].reset_index(drop=True)
        X_train = build_symbolic_features(train_df, columns=all_features)

        for coef in COEF_COLS:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)

            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1

        fold_count += 1

    rows = []
    for coef in COEF_COLS:
        for feat in all_features:
            rows.append({
                "coefficient": coef,
                "term": feat,
                "frequency": stability[coef][feat] / fold_count,
                "count": stability[coef][feat],
                "folds": fold_count,
            })
    return pd.DataFrame(rows)

stability_df = term_stability_table(coef_df)
stability_df.to_csv(os.path.join(OUTPUT_DIR, "term_stability.csv"), index=False)

stable_pivot = stability_df.pivot(index="coefficient", columns="term", values="frequency").fillna(0.0)

fig, ax = plt.subplots(figsize=(max(12, 0.45 * stable_pivot.shape[1]), 4))
im = ax.imshow(stable_pivot.values, aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(len(stable_pivot.index)))
ax.set_yticklabels(stable_pivot.index)
ax.set_xticks(range(len(stable_pivot.columns)))
ax.set_xticklabels(stable_pivot.columns, rotation=45, ha="right")
ax.set_title("Symbolic term stability across folds")
fig.colorbar(im, ax=ax, label="selection frequency")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_term_stability.png"), dpi=180, bbox_inches="tight")
plt.show()

display(stability_df.sort_values("frequency", ascending=False).head(25))

In [ ]:
STABILITY_THRESHOLD = 0.5

stable_terms_by_coef = {}
for coef in COEF_COLS:
    sub = stability_df[(stability_df["coefficient"] == coef) & (stability_df["frequency"] >= STABILITY_THRESHOLD)]
    stable_terms_by_coef[coef] = sub["term"].tolist()

stable_union_terms = sorted(set(sum(stable_terms_by_coef.values(), [])))

stable_terms_table = pd.DataFrame([
    {"coefficient": coef, "n_stable_terms": len(stable_terms_by_coef[coef]), "stable_terms": ", ".join(stable_terms_by_coef[coef])}
    for coef in COEF_COLS
])
display(stable_terms_table)
stable_terms_table.to_csv(os.path.join(OUTPUT_DIR, "stable_terms_by_coefficient.csv"), index=False)

## Final equations — full symbolic and pruned symbolic charts

In [ ]:
def fit_full_sparse_equations(df_in):
    equations = []
    preds = np.zeros((len(df_in), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        X = build_symbolic_features(df_in)
        y = df_in[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
        model = LassoCV(cv=min(5, len(df_in)), random_state=42, max_iter=30000)
        model.fit(Xs, y)
        preds[:, j] = model.predict(Xs)

        terms = [(feat, val) for feat, val in zip(X.columns, model.coef_) if abs(val) > 1e-4]
        equations.append({"chart": "full_symbolic", "coefficient": coef, "n_terms": len(terms), "equation": format_equation(coef, terms, model.intercept_)})

    return pd.DataFrame(equations), preds

def fit_pruned_equations(df_in, terms_by_coef):
    equations = []
    preds = np.zeros((len(df_in), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        terms = terms_by_coef.get(coef, [])
        y = df_in[coef].to_numpy(dtype=float)

        if len(terms) == 0:
            intercept = float(np.mean(y))
            preds[:, j] = intercept
            equations.append({"chart": "pruned_symbolic", "coefficient": coef, "n_terms": 0, "equation": f"{coef} = {intercept:.5f}"})
            continue

        X = build_symbolic_features(df_in, reduced_terms=terms)
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
        model = Ridge(alpha=1.0)
        model.fit(Xs, y)
        preds[:, j] = model.predict(Xs)

        active_terms = [(feat, val) for feat, val in zip(X.columns, model.coef_) if abs(val) > 1e-6]
        equations.append({"chart": "pruned_symbolic", "coefficient": coef, "n_terms": len(active_terms), "equation": format_equation(coef, active_terms, model.intercept_)})

    return pd.DataFrame(equations), preds

full_eq_df, full_fit_preds = fit_full_sparse_equations(coef_df)
pruned_eq_df, pruned_fit_preds = fit_pruned_equations(coef_df, stable_terms_by_coef)

equations_df = pd.concat([full_eq_df, pruned_eq_df], ignore_index=True)
display(equations_df)
equations_df.to_csv(os.path.join(OUTPUT_DIR, "final_symbolic_equations.csv"), index=False)

for _, row in pruned_eq_df.iterrows():
    print(row["equation"])
    print()

## Universality boundary and hard-block evaluation

In [ ]:
def predict_full_symbolic(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)

    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        model = LassoCV(cv=min(5, len(train_df)), random_state=42, max_iter=30000)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)

    return Yhat

def predict_pruned_symbolic(train_df, test_df, terms_by_coef):
    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)

    for j, coef in enumerate(COEF_COLS):
        terms = terms_by_coef.get(coef, [])
        if len(terms) == 0:
            Yhat[:, j] = train_df[coef].mean()
            continue

        X_train = build_symbolic_features(train_df, reduced_terms=terms)
        X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=terms)
        y_train = train_df[coef].to_numpy(dtype=float)

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)

        model = Ridge(alpha=1.0)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)

    return Yhat

def predict_ridge_chart(train_df, test_df):
    X_train = build_symbolic_features(train_df)
    X_test = build_symbolic_features(test_df, columns=X_train.columns)
    model = Ridge(alpha=1.0)
    model.fit(X_train, train_df[COEF_COLS].to_numpy(dtype=float))
    return model.predict(X_test)

hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "forcing_holdout::condition_gap": coef_df["forcing_mode"].astype(str) == "condition_gap",
    "forcing_holdout::capacity_gap": coef_df["forcing_mode"].astype(str) == "capacity_gap",
    "forcing_holdout::feature_gap": coef_df["forcing_mode"].astype(str) == "feature_gap",
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
    "flow_holdout::linear": coef_df["flow_mode"].astype(str) == "linear",
    "flow_holdout::nonlinear": coef_df["flow_mode"].astype(str) == "nonlinear",
}

rows = []

for block_name, test_mask in hard_block_masks.items():
    train_df = coef_df.loc[~test_mask].reset_index(drop=True)
    test_df = coef_df.loc[test_mask].reset_index(drop=True)

    if len(test_df) == 0 or len(train_df) < 5:
        continue

    predictions = {
        "full_symbolic": predict_full_symbolic(train_df, test_df),
        "pruned_symbolic": predict_pruned_symbolic(train_df, test_df, stable_terms_by_coef),
        "ridge_chart": predict_ridge_chart(train_df, test_df),
    }

    for i, rid in enumerate(test_df["regime_id"].tolist()):
        beta_true = test_df.loc[i, COEF_COLS].to_numpy(dtype=float)
        sub = regime_subsets[rid]
        y_emp = sub["predicted_flow"].to_numpy(dtype=float)

        for model_name, Yhat in predictions.items():
            beta_hat = Yhat[i]
            pred_field = predict_with_beta(sub, beta_hat)
            rows.append({
                "block": block_name,
                "regime_id": rid,
                "model": model_name,
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_emp, pred_field)),
                "traj_rmse": trajectory_gap(sub, beta_true, beta_hat),
            })

eval_df = pd.DataFrame(rows)
summary_df = eval_df.groupby(["block", "model"])[["coef_rmse", "field_rmse", "traj_rmse"]].mean().reset_index()
display(summary_df.sort_values(["block", "traj_rmse"]))
summary_df.to_csv(os.path.join(OUTPUT_DIR, "hard_block_summary.csv"), index=False)

In [ ]:
# Figure 3 — Universality boundary by block

for metric in ["coef_rmse", "field_rmse", "traj_rmse"]:
    pivot = summary_df.pivot(index="block", columns="model", values=metric)
    fig, ax = plt.subplots(figsize=(13, 5))
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"Universality boundary — {metric}")
    ax.set_ylabel(metric)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"fig3_boundary_{metric}.png"), dpi=180, bbox_inches="tight")
    plt.show()

## Minimal-chart tradeoff

Test stability thresholds to see how many terms are required before the chart saturates.

In [ ]:
threshold_rows = []

threshold_grid = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75]

for threshold in threshold_grid:
    terms_by_coef = {}
    for coef in COEF_COLS:
        sub = stability_df[(stability_df["coefficient"] == coef) & (stability_df["frequency"] >= threshold)]
        terms_by_coef[coef] = sub["term"].tolist()

    n_terms_total = sum(len(v) for v in terms_by_coef.values())

    block_scores = []
    for block_name, test_mask in hard_block_masks.items():
        train_df = coef_df.loc[~test_mask].reset_index(drop=True)
        test_df = coef_df.loc[test_mask].reset_index(drop=True)
        if len(test_df) == 0 or len(train_df) < 5:
            continue
        Yhat = predict_pruned_symbolic(train_df, test_df, terms_by_coef)

        for i, rid in enumerate(test_df["regime_id"].tolist()):
            beta_true = test_df.loc[i, COEF_COLS].to_numpy(dtype=float)
            sub = regime_subsets[rid]
            y_emp = sub["predicted_flow"].to_numpy(dtype=float)
            pred_field = predict_with_beta(sub, Yhat[i])
            block_scores.append({
                "block": block_name,
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, Yhat[i])),
                "field_rmse": math.sqrt(mean_squared_error(y_emp, pred_field)),
                "traj_rmse": trajectory_gap(sub, beta_true, Yhat[i]),
            })

    score_df = pd.DataFrame(block_scores)
    threshold_rows.append({
        "stability_threshold": threshold,
        "n_terms_total": n_terms_total,
        "mean_coef_rmse": score_df["coef_rmse"].mean(),
        "mean_field_rmse": score_df["field_rmse"].mean(),
        "mean_traj_rmse": score_df["traj_rmse"].mean(),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)
threshold_df.to_csv(os.path.join(OUTPUT_DIR, "minimal_chart_tradeoff.csv"), index=False)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(threshold_df["n_terms_total"], threshold_df["mean_traj_rmse"], marker="o")
for _, row in threshold_df.iterrows():
    ax.text(row["n_terms_total"], row["mean_traj_rmse"], f"{row['stability_threshold']:.2f}", fontsize=8)
ax.set_xlabel("total symbolic terms")
ax.set_ylabel("mean trajectory RMSE")
ax.set_title("Minimal-chart tradeoff")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_minimal_chart_tradeoff.png"), dpi=180, bbox_inches="tight")
plt.show()

## Family coefficient templates

In [ ]:
family_tables = []

for family_col in ["forcing_mode", "flow_mode", "system", "task"]:
    temp = coef_df.groupby(family_col)[COEF_COLS].mean()
    temp.to_csv(os.path.join(OUTPUT_DIR, f"template_by_{family_col}.csv"))
    family_tables.append((family_col, temp))

    fig, ax = plt.subplots(figsize=(9, 3.5))
    im = ax.imshow(temp.values, aspect="auto", cmap="coolwarm")
    ax.set_yticks(range(len(temp.index)))
    ax.set_yticklabels(temp.index)
    ax.set_xticks(range(len(COEF_COLS)))
    ax.set_xticklabels(COEF_COLS, rotation=30, ha="right")
    ax.set_title(f"Mean coefficient template by {family_col}")
    fig.colorbar(im, ax=ax, label="mean coefficient")
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"fig5_template_by_{family_col}.png"), dpi=180, bbox_inches="tight")
    plt.show()

## Representative field reconstruction figure

In [ ]:
# Pick a representative regime where pruned chart is strong

pruned_eval = eval_df[eval_df["model"] == "pruned_symbolic"].copy()
rep_row = pruned_eval.sort_values("traj_rmse").iloc[0]
rep_rid = rep_row["regime_id"]

# Refit pruned chart excluding this regime
train_df = coef_df[coef_df["regime_id"] != rep_rid].reset_index(drop=True)
test_df = coef_df[coef_df["regime_id"] == rep_rid].reset_index(drop=True)
beta_true = test_df[COEF_COLS].to_numpy(dtype=float)[0]
beta_pruned = predict_pruned_symbolic(train_df, test_df, stable_terms_by_coef)[0]
beta_full = predict_full_symbolic(train_df, test_df)[0]
beta_ridge = predict_ridge_chart(train_df, test_df)[0]

sub = regime_subsets[rep_rid]
y_emp = sub["predicted_flow"].to_numpy(dtype=float)

fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=True)
panels = [
    ("Empirical", y_emp),
    ("Pruned symbolic", predict_with_beta(sub, beta_pruned)),
    ("Full symbolic", predict_with_beta(sub, beta_full)),
    ("Ridge chart", predict_with_beta(sub, beta_ridge)),
]

for ax, (title, vals) in zip(axes, panels):
    sc = ax.scatter(sub["condition_coord"], sub["residual"], c=vals, s=16)
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel("residual r")
fig.suptitle(f"Representative field reconstruction: {rep_rid}", y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig6_representative_field_reconstruction.png"), dpi=180, bbox_inches="tight")
plt.show()

## Final paper summary and verdict table

In [ ]:
decision_rows = []

for block, sub in summary_df.groupby("block"):
    best = sub.sort_values("traj_rmse").iloc[0]
    pruned = sub[sub["model"] == "pruned_symbolic"].iloc[0]
    full = sub[sub["model"] == "full_symbolic"].iloc[0]
    ridge = sub[sub["model"] == "ridge_chart"].iloc[0]
    pruned_to_best = pruned["traj_rmse"] / max(best["traj_rmse"], 1e-12)

    if best["model"] == "pruned_symbolic":
        verdict = "minimal symbolic chart wins"
    elif pruned_to_best <= 1.25:
        verdict = "minimal symbolic chart competitive"
    elif best["model"] == "full_symbolic":
        verdict = "full symbolic chart needed"
    else:
        verdict = "regularized dense chart wins"

    decision_rows.append({
        "block": block,
        "best_model": best["model"],
        "best_traj_rmse": best["traj_rmse"],
        "pruned_traj_rmse": pruned["traj_rmse"],
        "full_traj_rmse": full["traj_rmse"],
        "ridge_traj_rmse": ridge["traj_rmse"],
        "pruned_to_best_ratio": pruned_to_best,
        "verdict": verdict,
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)
decision_df.to_csv(os.path.join(OUTPUT_DIR, "final_verdict_table.csv"), index=False)

In [ ]:
# Generate Markdown summary artifact

summary_md = f"""# Zeta Constraint Lab — Notebook 63 Synthesis

## Final model

Shared field:

```text
g(r,c;β) = β0 + βc c + βr r + βc3 c^3 + βr2 r^2 + βrc2 r c^2
```

Symbolic chart:

```text
β = f(forcing_mode, flow_mode, system, task, k)
```

## Data source

- `{data_source}`
- Synthetic fallback: `{USE_SYNTHETIC_FALLBACK}`
- Regime count: `{len(coef_df)}`
- Coefficient terms: `{', '.join(COEF_COLS)}`

## PCA manifold

Explained variance:

```text
{np.array2string(pca.explained_variance_ratio_, precision=4)}
```

Cumulative explained variance:

```text
{np.array2string(np.cumsum(pca.explained_variance_ratio_), precision=4)}
```

## Stable symbolic terms

{stable_terms_table.to_markdown(index=False)}

## Final decision table

{decision_df.to_markdown(index=False)}

## Main conclusion

The residual-flow field is stable in `(r,c)`, while regime variation is captured by a sparse symbolic coordinate chart for the coefficient vector. Smooth latent ODE transport and quadratic latent transport are useful diagnostics but are not the final predictive abstraction. The final model is a shared governing field plus a symbolic coefficient chart, with universality boundaries exposed by forcing/system/flow holdouts.
"""

with open(os.path.join(OUTPUT_DIR, "paper_synthesis_summary.md"), "w", encoding="utf-8") as f:
    f.write(summary_md)

display(Markdown(summary_md[:4000] + "\n\n..."))

## Working conclusion

Notebook 63 consolidates the full zeta-constraint-lab arc:

1. The residual flow admits a shared low-order governing template in `(r,c)`.
2. Regime dependence is concentrated in six coefficient coordinates.
3. Those coefficients form a low-dimensional manifold.
4. Smooth latent ODE transport does not beat direct symbolic prediction.
5. A sparse symbolic coordinate chart is the final predictive abstraction.
6. Universality boundaries are exposed by harder holdout blocks.

Recommended next step:

**Notebook 64 — arXiv-style paper assembly and figure export**